## 1. Khởi tạo Spark

In [1]:
import os, sys

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
# Spark trên Windows cần hadoop.dll trong PATH để ghi file
os.environ["PATH"] = os.path.join(os.environ.get("HADOOP_HOME", r"C:\hadoop"), "bin") + os.pathsep + os.environ["PATH"]

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType, LongType,
                               DoubleType, DateType)

# Cho phép import các module .py trong thư mục src
sys.path.insert(0, os.path.abspath("../src"))
from geo_utils import haversine_km
from features import add_features

spark = (SparkSession.builder
         .master("local[*]")
         .appName("RussiaRealEstatePreprocessing")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())
spark.version

'3.5.3'

## 2. Đọc dữ liệu

Tập train là `data/clean.parquet` (notebook 01); tập test là bộ Russia Real Estate 2021, chỉ 2 tháng đầu sau giai đoạn train (02/05–30/06/2021).

In [2]:
train = spark.read.parquet("../data/clean.parquet").withColumn("date", F.to_date("date"))

test_schema = StructType([
    StructField("date",          DateType()),
    StructField("price",         LongType()),
    StructField("level",         IntegerType()),
    StructField("levels",        IntegerType()),
    StructField("rooms",         IntegerType()),
    StructField("area",          DoubleType()),
    StructField("kitchen_area",  DoubleType()),
    StructField("geo_lat",       DoubleType()),
    StructField("geo_lon",       DoubleType()),
    StructField("building_type", IntegerType()),
    StructField("object_type",   IntegerType()),
    StructField("postal_code",   IntegerType()),
    StructField("street_id",     IntegerType()),
    StructField("id_region",     IntegerType()),
    StructField("house_id",      IntegerType()),
])
TEST_PATH = "../data/raw/russia-real-estate-2021/russia-real-estate-2021-from-2021-05-02-to-2021-06-30.csv"
test_raw = spark.read.csv(TEST_PATH, sep=";", header=True, schema=test_schema)
print(f"Train: {train.count():,} bản ghi | Test: {test_raw.count():,} bản ghi")

Train: 5,470,541 bản ghi | Test: 1,739,965 bản ghi


## 3. Làm sạch tập test

Áp dụng các quy tắc như notebook 01 (bỏ trùng trước khi bỏ cột, `price > 0`, `level <= levels`), thêm điều kiện `level`, `levels` dương; không lọc theo `kitchen_area` vì cột này không dùng cho mô hình.

In [3]:
test = test_raw.dropDuplicates().cache()
n_raw, n = test_raw.count(), test.count()
print(f"{'Trùng hoàn toàn':<22} loại {n_raw - n:>7,} bản ghi")

keep_rules = {
    "price <= 0":          F.col("price") > 0,
    "level, levels <= 0":  (F.col("level") > 0) & (F.col("levels") > 0),
    "level > levels":      F.col("level") <= F.col("levels"),
}
for name, keep in keep_rules.items():
    n_prev, test = n, test.filter(keep)
    n = test.count()
    print(f"{name:<22} loại {n_prev - n:>7,} bản ghi")
print(f"Còn lại: {n:,} / {n_raw:,} bản ghi ({n / n_raw:.2%})")

Trùng hoàn toàn        loại  50,757 bản ghi


price <= 0             loại   4,329 bản ghi


level, levels <= 0     loại     342 bản ghi


level > levels         loại      80 bản ghi
Còn lại: 1,684,457 / 1,739,965 bản ghi (96.81%)


## 4. Đổi cấu trúc tập test cho khớp tập train

`id_region` của tập test là số hiệu vùng chính thức của Nga; đổi sang tên vùng bằng bảng đối chiếu, rồi sang mã `region` và tọa độ thủ phủ của tập train (`data/raw/region_names.csv`). Vùng không có trong tập train (Chukotka) bị loại.

In [4]:
# Số hiệu vùng chính thức -> tên vùng như trong region_names.csv; mã 200 có tọa độ trung vị ở Surgut nên gán cho Khanty-Mansi (suy luận từ tọa độ)
OFFICIAL_CODES = {
    1: "Republic of Adygea", 2: "Republic of Bashkortostan", 3: "Republic of Buryatia", 4: "Altai Republic",
    5: "Republic of Dagestan", 6: "Republic of Ingushetia", 7: "Kabardino-Balkar Republic", 8: "Republic of Kalmykia",
    9: "Karachay-Cherkess Republic", 10: "Republic of Karelia", 11: "Komi Republic", 12: "Mari El Republic",
    13: "Republic of Mordovia", 14: "Sakha Republic", 15: "Republic of North Ossetia-Alania", 16: "Republic of Tatarstan",
    17: "Tuva Republic", 18: "Udmurt Republic", 19: "Republic of Khakassia", 20: "Chechen Republic",
    21: "Chuvash Republic", 22: "Altai Krai", 23: "Krasnodar Krai", 24: "Krasnoyarsk Krai",
    25: "Primorsky Krai", 26: "Stavropol Krai", 27: "Khabarovsk Krai", 28: "Amur Oblast",
    29: "Arkhangelsk Oblast", 30: "Astrakhan Oblast", 31: "Belgorod Oblast", 32: "Bryansk Oblast",
    33: "Vladimir Oblast", 34: "Volgograd Oblast", 35: "Vologda Oblast", 36: "Voronezh Oblast",
    37: "Ivanovo Oblast", 38: "Irkutsk Oblast", 39: "Kaliningrad Oblast", 40: "Kaluga Oblast",
    41: "Kamchatka Krai", 42: "Kemerovo Oblast", 43: "Kirov Oblast", 44: "Kostroma Oblast",
    45: "Kurgan Oblast", 46: "Kursk Oblast", 47: "Leningrad Oblast", 48: "Lipetsk Oblast",
    49: "Magadan Oblast", 50: "Moscow Oblast", 51: "Murmansk Oblast", 52: "Nizhny Novgorod Oblast",
    53: "Novgorod Oblast", 54: "Novosibirsk Oblast", 55: "Omsk Oblast", 56: "Orenburg Oblast",
    57: "Oryol Oblast", 58: "Penza Oblast", 59: "Perm Krai", 60: "Pskov Oblast",
    61: "Rostov Oblast", 62: "Ryazan Oblast", 63: "Samara Oblast", 64: "Saratov Oblast",
    65: "Sakhalin Oblast", 66: "Sverdlovsk Oblast", 67: "Smolensk Oblast", 68: "Tambov Oblast",
    69: "Tver Oblast", 70: "Tomsk Oblast", 71: "Tula Oblast", 72: "Tyumen Oblast",
    73: "Ulyanovsk Oblast", 74: "Chelyabinsk Oblast", 75: "Zabaykalsky Krai", 76: "Yaroslavl Oblast",
    77: "Moscow", 78: "Saint Petersburg", 79: "Jewish Autonomous Oblast", 83: "Nenets Autonomous Okrug",
    86: "Khanty-Mansi Autonomous Okrug", 87: "Chukotka Autonomous Okrug", 89: "Yamalo-Nenets Autonomous Okrug",
    91: "Republic of Crimea", 92: "Sevastopol", 200: "Khanty-Mansi Autonomous Okrug",
}
code_to_province = F.create_map(*[F.lit(x) for kv in OFFICIAL_CODES.items() for x in kv])
regions = (spark.read.csv("../data/raw/region_names.csv", header=True, inferSchema=True)
           .withColumn("center", F.split("capital_coord", ", "))
           .select("region", "province",
                   F.col("center")[0].cast("double").alias("center_lat"),
                   F.col("center")[1].cast("double").alias("center_lon")))

n_before = test.count()
test = (test.withColumn("province", code_to_province[F.col("id_region")])
            .join(F.broadcast(regions), on="province", how="inner"))
print(f"Loại do vùng không có trong tập train hoặc mã không có trong bảng: {n_before - test.count():,} bản ghi")

Loại do vùng không có trong tập train hoặc mã không có trong bảng: 16 bản ghi


Kiểm tra bảng đối chiếu: khoảng cách từ tọa độ trung vị của mỗi mã tới thủ phủ của vùng được gán (khoảng cách lớn là dấu hiệu gán sai).

In [5]:
check = (test.groupBy("id_region", "province", "center_lat", "center_lon")
             .agg(F.count("*").alias("count"),
                  F.percentile_approx("geo_lat", 0.5).alias("lat"), F.percentile_approx("geo_lon", 0.5).alias("lon"))
             .withColumn("km_to_capital", haversine_km(F.col("lat"), F.col("lon"), F.col("center_lat"), F.col("center_lon")))
             .select("id_region", "province", "count", "km_to_capital")
             .orderBy(F.desc("km_to_capital"))
             .toPandas())
print(f"Số mã: {len(check)} | khoảng cách trung vị: {check['km_to_capital'].median():.1f} km")
print(check.head(10).to_string(index=False, float_format="{:,.1f}".format))

Số mã: 85 | khoảng cách trung vị: 2.6 km
 id_region                       province  count  km_to_capital
        89 Yamalo-Nenets Autonomous Okrug    987          451.4
        11                  Komi Republic   6725          237.6
        86  Khanty-Mansi Autonomous Okrug  18036          237.3
         1             Republic of Adygea   3248          103.5
        47               Leningrad Oblast  26185           14.0
         9     Karachay-Cherkess Republic    563           10.2
        50                  Moscow Oblast 106299            9.0
        10            Republic of Karelia   2126            8.4
        48                 Lipetsk Oblast   5980            6.8
        23                 Krasnodar Krai 111210            6.4


Đổi các cột còn lại theo quy ước của tập train: `building_type` sang nhãn (mã 0 "không rõ" thành `Unknown`), `object_type` sang `pre_owned_house`, studio (`rooms = -1`) sang `is_studio` và `rooms = 0`, `kitchen_area <= 0` coi là thiếu; tính `distance_to_capital`.

In [6]:
TEST_BUILDING_TYPES = {0: "Unknown", 1: "Others", 2: "Panel", 3: "Monolithic", 4: "Brick", 5: "Block", 6: "Wooden"}
bt_map = F.create_map(*[F.lit(x) for kv in TEST_BUILDING_TYPES.items() for x in kv])

test = (test
    .withColumn("building_type", bt_map[F.col("building_type")])
    .withColumn("pre_owned_house", (F.col("object_type") == 0).cast("int"))
    .withColumn("is_studio", (F.col("rooms") == -1).cast("int"))
    .withColumn("rooms", F.when(F.col("rooms") == -1, 0).otherwise(F.col("rooms")))
    .withColumn("kitchen_area", F.when(F.col("kitchen_area") > 0, F.col("kitchen_area")))
    .withColumn("distance_to_capital", haversine_km(F.col("geo_lat"), F.col("geo_lon"),
                                                    F.col("center_lat"), F.col("center_lon"))))

COLUMNS = ["price", "date", "geo_lat", "geo_lon", "region", "province", "distance_to_capital", "building_type",
           "pre_owned_house", "level", "levels", "rooms", "is_studio", "area", "kitchen_area"]
test = test.select(COLUMNS)
train = train.select(COLUMNS)
test.groupBy("building_type").count().orderBy(F.desc("count")).show()

+-------------+------+
|building_type| count|
+-------------+------+
|      Unknown|813433|
|        Brick|330133|
|        Panel|284001|
|   Monolithic|167322|
|       Others| 47295|
|        Block| 36876|
|       Wooden|  5381|
+-------------+------+



## 5. Cắt ngoại lai của `price`

Ngưỡng tính trên tập train theo nhóm vùng, tại điểm gãy trên biểu đồ phân vị (notebook 02): Moscow cắt 0,2% đầu dưới và 0,1% đầu trên, Saint Petersburg và các vùng còn lại cắt 0,1% mỗi đầu; áp dụng cùng ngưỡng cho tập test.

In [7]:
price_group = F.when(F.col("province").isin("Moscow", "Saint Petersburg"), F.col("province")).otherwise("Phần còn lại")
CUTS = {"Moscow": (0.002, 0.001), "Saint Petersburg": (0.001, 0.001), "Phần còn lại": (0.001, 0.001)}  # (dưới, trên)
levels = sorted({round(p, 6) for lo, hi in CUTS.values() for p in (lo, 1 - hi)})
q = {r["g"]: dict(zip(levels, r["q"]))
     for r in train.groupBy(price_group.alias("g")).agg(F.percentile("price", levels).alias("q")).collect()}
bounds = {g: (q[g][round(lo, 6)], q[g][round(1 - hi, 6)]) for g, (lo, hi) in CUTS.items()}

def by_group(values):
    """Biểu thức Spark trả về values[nhóm vùng] của từng bản ghi."""
    expr = None
    for g, v in values.items():
        expr = (F.when if expr is None else expr.when)(price_group == g, F.lit(v))
    return expr

in_bounds = F.col("price").between(by_group({g: b[0] for g, b in bounds.items()}),
                                   by_group({g: b[1] for g, b in bounds.items()}))
summary = {}
for name, data in [("train", train), ("test", test)]:
    counts = data.groupBy(price_group.alias("g")).agg(F.count("*").alias("n"), F.sum((~in_bounds).cast("int")).alias("removed")).toPandas()
    summary[name] = counts.set_index("g")
train, test = train.filter(in_bounds), test.filter(in_bounds)

cut_table = pd.DataFrame({"lower": {g: b[0] for g, b in bounds.items()}, "upper": {g: b[1] for g, b in bounds.items()}})
for name, s in summary.items():
    cut_table[f"{name}_removed"] = s["removed"]
    cut_table[f"{name}_removed_pct"] = (100 * s["removed"] / s["n"]).round(2)
print(cut_table.to_string(float_format="{:,.2f}".format))

                        lower          upper  train_removed  train_removed_pct  test_removed  test_removed_pct
Moscow           1,150,000.00 241,480,345.00           1294               0.29           847              0.63
Saint Petersburg 1,100,000.00 120,000,000.00            859               0.19           143              0.13
Phần còn lại       200,000.00  26,000,000.00           8794               0.19          5970              0.41


## 6. Bỏ bản ghi test trùng với tập train

Bỏ các bản ghi của tập test trùng với tập train ở `price`, tọa độ, `area`, `level`, `levels`, `rooms`, `is_studio` và `pre_owned_house` (tin được rao lại y hệt sau mốc 01/05/2021), để tập test chỉ gồm các bản ghi mô hình chưa thấy khi huấn luyện; không so `building_type` và `kitchen_area` vì hai cột này thường bị thiếu ở tập test.

In [8]:
DUP_KEYS = ["price", "geo_lat", "geo_lon", "area", "level", "levels", "rooms", "is_studio", "pre_owned_house"]
n_before = test.count()
test = test.join(train.select(DUP_KEYS).distinct(), on=DUP_KEYS, how="left_anti").select(test.columns)  # giữ thứ tự cột
n_after = test.count()
print(f"Loại {n_before - n_after:,} bản ghi test trùng với tập train ({(n_before - n_after) / n_before:.2%}), còn {n_after:,} bản ghi")

Loại 104,486 bản ghi test trùng với tập train (6.23%), còn 1,572,995 bản ghi


## 7. Tạo đặc trưng

Dùng chung hàm `add_features` (`src/features.py`) cho cả hai tập: `year`, `month`, `months_since_start` (mốc 2018-02-01), `is_moscow_or_spb`, `level_ratio` (= `level / levels`).

In [9]:
train, test = add_features(train), add_features(test)
train.select("date", "months_since_start", "is_moscow_or_spb", "level_ratio").show(5)
test.agg(F.min("months_since_start"), F.max("months_since_start")).show()

+----------+------------------+----------------+------------------+
|      date|months_since_start|is_moscow_or_spb|       level_ratio|
+----------+------------------+----------------+------------------+
|2018-03-01|                 1|               0|              0.75|
|2018-04-07|                 2|               1|               0.5|
|2018-04-30|                 2|               0|               0.6|
|2018-05-14|                 3|               0|0.6666666666666666|
|2018-05-28|                 3|               0|               0.8|
+----------+------------------+----------------+------------------+
only showing top 5 rows



+-----------------------+-----------------------+
|min(months_since_start)|max(months_since_start)|
+-----------------------+-----------------------+
|                     39|                     40|
+-----------------------+-----------------------+



## 8. Lưu dữ liệu

Lưu `data/train.parquet` và `data/test.parquet`. `building_type` và `kitchen_area` vẫn giữ trong dữ liệu để tham khảo nhưng không dùng cho mô hình.

In [10]:
train.write.mode("overwrite").parquet("../data/train.parquet")
test.write.mode("overwrite").parquet("../data/test.parquet")
for name in ["train", "test"]:
    print(f"{name}: {spark.read.parquet(f'../data/{name}.parquet').count():,} bản ghi")
spark.read.parquet("../data/test.parquet").printSchema()

train: 5,459,594 bản ghi


test: 1,572,995 bản ghi
root
 |-- price: long (nullable = true)
 |-- date: date (nullable = true)
 |-- geo_lat: double (nullable = true)
 |-- geo_lon: double (nullable = true)
 |-- region: integer (nullable = true)
 |-- province: string (nullable = true)
 |-- distance_to_capital: double (nullable = true)
 |-- building_type: string (nullable = true)
 |-- pre_owned_house: integer (nullable = true)
 |-- level: integer (nullable = true)
 |-- levels: integer (nullable = true)
 |-- rooms: integer (nullable = true)
 |-- is_studio: integer (nullable = true)
 |-- area: double (nullable = true)
 |-- kitchen_area: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- months_since_start: integer (nullable = true)
 |-- is_moscow_or_spb: integer (nullable = true)
 |-- level_ratio: double (nullable = true)



5 bản ghi đầu của tập train và tập test sau tiền xử lý.

In [11]:
for name in ["train", "test"]:
    print(name)
    spark.read.parquet(f"../data/{name}.parquet").show(5)

train


+-------+----------+----------+----------+------+------------------+-------------------+-------------+---------------+-----+------+-----+---------+----+------------+----+-----+------------------+----------------+------------------+
|  price|      date|   geo_lat|   geo_lon|region|          province|distance_to_capital|building_type|pre_owned_house|level|levels|rooms|is_studio|area|kitchen_area|year|month|months_since_start|is_moscow_or_spb|       level_ratio|
+-------+----------+----------+----------+------+------------------+-------------------+-------------+---------------+-----+------+-----+---------+----+------------+----+-----+------------------+----------------+------------------+
|4600000|2018-04-02|55.7078344|37.9596388|    81|     Moscow Oblast|  22.05936308378456|   Monolithic|              1|   17|    17|    1|        0|37.0|        12.5|2018|    4|                 2|               0|               1.0|
|2800000|2018-04-05|55.9428756|37.9764623|    81|     Moscow Oblast|  30

+------+----------+----------+----------+------+--------------------+-------------------+-------------+---------------+-----+------+-----+---------+----+------------+----+-----+------------------+----------------+-----------+
| price|      date|   geo_lat|   geo_lon|region|            province|distance_to_capital|building_type|pre_owned_house|level|levels|rooms|is_studio|area|kitchen_area|year|month|months_since_start|is_moscow_or_spb|level_ratio|
+------+----------+----------+----------+------+--------------------+-------------------+-------------+---------------+-----+------+-----+---------+----+------------+----+-----+------------------+----------------+-----------+
|200000|2021-06-19|55.0717675|71.0529512|  8640|         Omsk Oblast| 149.70245836426835|        Brick|              1|    1|     1|    3|        0|52.0|        NULL|2021|    6|                40|               0|        1.0|
|200000|2021-05-07|56.1298527|88.5317549|  2860|     Kemerovo Oblast| 176.28886893009425|       

## 9. Dữ liệu đưa vào mô hình

Đọc lại `train.parquet` và `test.parquet` để mục này chạy độc lập với các mục trên.

In [12]:
from pyspark.sql import Window

train = spark.read.parquet("../data/train.parquet")
test = spark.read.parquet("../data/test.parquet")

Điền `building_type = Unknown` của tập test bằng mode của tập train theo nhóm (`province`, `pre_owned_house`, `is_studio`); nhóm không có trong tập train thì lùi dần về (`province`, `pre_owned_house`), `province`, rồi mode chung.

In [13]:
def mode_by(cols, name):
    """Loại tường phổ biến nhất của tập train trong từng nhóm (bằng nhau thì lấy theo thứ tự chữ cái)."""
    w = Window.partitionBy(*cols).orderBy(F.desc("count"), F.asc("building_type"))
    return (train.groupBy(*cols, "building_type").count()
                 .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
                 .select(*cols, F.col("building_type").alias(name)))

keys = ["province", "pre_owned_house", "is_studio"]
global_mode = train.groupBy("building_type").count().orderBy(F.desc("count")).first()["building_type"]
test = (test.join(F.broadcast(mode_by(keys, "mode_3")), keys, "left")
            .join(F.broadcast(mode_by(keys[:2], "mode_2")), keys[:2], "left")
            .join(F.broadcast(mode_by(keys[:1], "mode_1")), keys[:1], "left"))
unknown = F.col("building_type") == "Unknown"
print(test.agg(F.sum(unknown.cast("int")).alias("Unknown"),
               F.sum((unknown & F.col("mode_3").isNotNull()).cast("int")).alias("điền theo 3 cột"),
               F.sum((unknown & F.col("mode_3").isNull() & F.col("mode_2").isNotNull()).cast("int")).alias("theo 2 cột"),
               F.sum((unknown & F.col("mode_3").isNull() & F.col("mode_2").isNull()).cast("int")).alias("theo province hoặc chung"))
          .toPandas().to_string(index=False))
test = (test.withColumn("building_type", F.when(unknown, F.coalesce("mode_3", "mode_2", "mode_1", F.lit(global_mode)))
                                          .otherwise(F.col("building_type")))
            .drop("mode_3", "mode_2", "mode_1"))
test.groupBy("building_type").count().orderBy(F.desc("count")).show()

 Unknown  điền theo 3 cột  theo 2 cột  theo province hoặc chung
  786729           786608         119                         2


+-------------+------+
|building_type| count|
+-------------+------+
|        Brick|656737|
|        Panel|450733|
|   Monolithic|368991|
|       Others| 51391|
|        Block| 40127|
|       Wooden|  5016|
+-------------+------+



Mã hóa one-hot `building_type` với các nhóm của tập train, bỏ Panel (nhóm đông nhất) làm nhóm tham chiếu để tránh đa cộng tuyến hoàn hảo; giữ các cột dùng cho mô hình cùng `province` và bỏ `year`, `month`, `date`, `kitchen_area`, `levels`, `region`.

In [14]:
BT_REFERENCE = "Panel"
bt_cats = sorted(r["building_type"] for r in train.select("building_type").distinct().collect())
bt_cols = [f"bt_{c.lower()}" for c in bt_cats if c != BT_REFERENCE]
MODEL_COLUMNS = (["price", "geo_lat", "geo_lon", "distance_to_capital", "area", "rooms", "is_studio", "level", "level_ratio",
                  "pre_owned_house", "months_since_start", "is_moscow_or_spb", "province"] + bt_cols)

def to_model_data(df):
    for c in bt_cats:
        if c != BT_REFERENCE:
            df = df.withColumn(f"bt_{c.lower()}", (F.col("building_type") == c).cast("int"))
    return df.select(MODEL_COLUMNS)

train_model, test_model = to_model_data(train), to_model_data(test)
print("Nhóm building_type:", bt_cats, "| tham chiếu:", BT_REFERENCE)
train_model.printSchema()

Nhóm building_type: ['Block', 'Brick', 'Monolithic', 'Others', 'Panel', 'Wooden'] | tham chiếu: Panel
root
 |-- price: long (nullable = true)
 |-- geo_lat: double (nullable = true)
 |-- geo_lon: double (nullable = true)
 |-- distance_to_capital: double (nullable = true)
 |-- area: double (nullable = true)
 |-- rooms: integer (nullable = true)
 |-- is_studio: integer (nullable = true)
 |-- level: integer (nullable = true)
 |-- level_ratio: double (nullable = true)
 |-- pre_owned_house: integer (nullable = true)
 |-- months_since_start: integer (nullable = true)
 |-- is_moscow_or_spb: integer (nullable = true)
 |-- province: string (nullable = true)
 |-- bt_block: integer (nullable = true)
 |-- bt_brick: integer (nullable = true)
 |-- bt_monolithic: integer (nullable = true)
 |-- bt_others: integer (nullable = true)
 |-- bt_wooden: integer (nullable = true)



Lưu `data/train_model.parquet` và `data/test_model.parquet`, rồi xem 5 bản ghi đầu.

In [15]:
train_model.write.mode("overwrite").parquet("../data/train_model.parquet")
test_model.write.mode("overwrite").parquet("../data/test_model.parquet")
for name in ["train_model", "test_model"]:
    saved = spark.read.parquet(f"../data/{name}.parquet")
    print(f"{name}: {saved.count():,} bản ghi, {len(saved.columns)} cột")
    saved.show(5)

train_model: 5,459,594 bản ghi, 18 cột


+-------+----------+----------+-------------------+----+-----+---------+-----+------------------+---------------+------------------+----------------+------------------+--------+--------+-------------+---------+---------+
|  price|   geo_lat|   geo_lon|distance_to_capital|area|rooms|is_studio|level|       level_ratio|pre_owned_house|months_since_start|is_moscow_or_spb|          province|bt_block|bt_brick|bt_monolithic|bt_others|bt_wooden|
+-------+----------+----------+-------------------+----+-----+---------+-----+------------------+---------------+------------------+----------------+------------------+--------+--------+-------------+---------+---------+
|4600000|55.7078344|37.9596388|  22.05936308378456|37.0|    1|        0|   17|               1.0|              1|                 2|               0|     Moscow Oblast|       0|       0|            1|        0|        0|
|2800000|55.9428756|37.9764623|  30.55904459640708|47.0|    2|        0|    5|               1.0|              1|   

test_model: 1,572,995 bản ghi, 18 cột
+------+----------+----------+-------------------+----+-----+---------+-----+-----------+---------------+------------------+----------------+------------------+--------+--------+-------------+---------+---------+
| price|   geo_lat|   geo_lon|distance_to_capital|area|rooms|is_studio|level|level_ratio|pre_owned_house|months_since_start|is_moscow_or_spb|          province|bt_block|bt_brick|bt_monolithic|bt_others|bt_wooden|
+------+----------+----------+-------------------+----+-----+---------+-----+-----------+---------------+------------------+----------------+------------------+--------+--------+-------------+---------+---------+
|200000|55.3469242|85.8899188| 12.673829093649806|44.0|    2|        0|    2|        0.4|              1|                40|               0|   Kemerovo Oblast|       0|       0|            0|        0|        0|
|200000|58.1700551|59.7097067|  157.8888959687897|33.0|    1|        0|    1|        1.0|              1|     